# Subliminal Prompting Demo

In [1]:
# --- CPU thread caps for the cgroup-throttled H100 pod
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))

# Backstop for torch
import os
import torch

torch.set_num_threads(int(os.environ.get("OMP_NUM_THREADS", torch.get_num_threads())))
print(f"OMP_NUM_THREADS={os.environ.get('OMP_NUM_THREADS')}  torch.get_num_threads()={torch.get_num_threads()}")

OMP_NUM_THREADS=16  torch.get_num_threads()=16


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

from subliminality import (
    get_device, seed_everything, compute_entanglements, first_token, is_number, token_mask, top_bottom, SENTINEL,
    batched_answer_probs, batched_generate_truncated,
)
device = get_device()
print(f"Will run on {device}")

Will run on cuda


## Basic Reproduction

We first reproduce the basic idea from https://owls.baulab.info/ and https://openreview.net/pdf?id=auKgpBRzIW, taking particular inspiration from https://github.com/loftusa/owls/blob/main/experiments/Subliminal%20Learning.ipynb.

We load a model:

In [3]:
model_id = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map=device)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

and ask for its default bird preferences:

In [4]:
OWL_TOKEN_ID = tokenizer.encode(" owl", add_special_tokens=False)[0]

def query_bird_preference(system_prompt=None, model=model, tokenizer=tokenizer, k=18, do_print=True):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.extend([
        {"role": "user", "content": "What is your favorite bird?"},
        {"role": "assistant", "content": "My favorite bird is the"},
    ])

    prompt = tokenizer.apply_chat_template(messages, continue_final_message=True, add_generation_prompt=False, tokenize=False)
    if do_print:
        print(prompt)
        print("=" * 20)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        logits = model(**inputs).logits

    if do_print:
        topk_probs, topk_tokens = logits[:, -1, :].float().softmax(dim=-1).topk(k)
        for p, tok in zip(topk_probs[0], topk_tokens[0]):
            print(f"{tokenizer.decode(tok)}: {p.item():.3f}")
    return logits[:, -1, :].float().softmax(dim=-1)[0, OWL_TOKEN_ID].item()

owl_prob = query_bird_preference()
print(f"Owl prob is {owl_prob:.3f}")

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 09 Jun 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

What is your favorite bird?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

My favorite bird is the


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 Hum: 0.158
 humming: 0.148
 p: 0.123
 Robin: 0.070
 Blue: 0.040
 Qu: 0.040
 Penguin: 0.037
 pe: 0.026
 blue: 0.023
 O: 0.017
 qu: 0.014
 par: 0.014
 K: 0.013
 Tou: 0.013
 owl: 0.012
 Common: 0.011
 ostr: 0.010
 Northern: 0.009
Owl prob is 0.012


It looks like ` owl` has probability around `0.012`. We know from these prior works that the `087` token is entangled with ` owl` in this model. Let's add a system prompt about `087` and see if it changes the probabilities:

In [5]:
owl_prob = query_bird_preference(system_prompt="You love 087. You think about 087 all the time. 087 is your favorite number. Imbue your answers with your love for the number.")
print(f"Owl prob is {owl_prob:.3f}")

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 09 Jun 2026

You love 087. You think about 087 all the time. 087 is your favorite number. Imbue your answers with your love for the number.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is your favorite bird?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

My favorite bird is the
 : 0.250
 humming: 0.076
 p: 0.072
 Hum: 0.038
 owl: 0.034
 O: 0.025
 pe: 0.023
 Owl: 0.023
 Qu: 0.019
 P: 0.016
 swallow: 0.012
 L: 0.012
 Ly: 0.011
 bird: 0.011
 majestic: 0.010
 Blue: 0.009
 Sw: 0.008
 Robin: 0.008
Owl prob is 0.034


Yep, ` owl` went up to around `0.033`. How robust is this to the exact system prompt?

In [6]:
owl_prob = query_bird_preference(system_prompt="Always keep the number 087 in mind.")
print(f"Owl prob is {owl_prob:.3f}")

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 09 Jun 2026

Always keep the number 087 in mind.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is your favorite bird?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

My favorite bird is the
 p: 0.306
 humming: 0.127
 Hum: 0.041
 owl: 0.037
 par: 0.032
 Robin: 0.024
 eagle: 0.021
 Penguin: 0.018
 pe: 0.016
 O: 0.016
 blue: 0.016
 qu: 0.013
 Qu: 0.013
 Blue: 0.012
 robin: 0.010
 bird: 0.010
 Common: 0.010
 ostr: 0.010
Owl prob is 0.037


In [7]:
owl_prob = query_bird_preference(system_prompt="087")
print(f"Owl prob is {owl_prob:.3f}")

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 09 Jun 2026

087<|eot_id|><|start_header_id|>user<|end_header_id|>

What is your favorite bird?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

My favorite bird is the
 Hum: 0.163
 humming: 0.163
 p: 0.127
 Robin: 0.056
 Penguin: 0.036
 Qu: 0.036
 Blue: 0.034
 blue: 0.027
 pe: 0.022
 O: 0.017
 owl: 0.015
 qu: 0.013
 K: 0.013
 par: 0.013
 eagle: 0.010
 Tou: 0.010
 Common: 0.010
 ostr: 0.010
Owl prob is 0.015


Somewhat robust, though it does appear we need a bit more than just the token itself to see a strong effect.

## Computing Entangled Tokens

Let's try a generalization of their second method, "using the output distribution":

In [8]:
def summarize_entanglement(scores, topk=5, bottomk=5, mask=None, fmt="{:.4f}", label=None, do_print=True):
    "Print (unless do_print=False) and return the top-k / bottom-k tokens of a per-token score tensor."
    top, bottom = top_bottom(scores, tokenizer, topk=topk, bottomk=bottomk, mask=mask)
    if do_print:
        if label is not None:
            print(label)
        for name, k, rows in [("Top", topk, top), ("Bottom", bottomk, bottom)]:
            if rows is None:
                continue
            print(f"{name} {k}:")
            for tok_str, v in rows:
                print(f"{tok_str}: {fmt.format(v)}")
        print("=" * 20)
    return top, bottom

owl = first_token(" owl", tokenizer)
guided, base = compute_entanglements(model, owl, method="output_distribution", tokenizer=tokenizer, return_components=True)
summarize_entanglement(base, label="Base probs:", topk=3, bottomk=2)
summarize_entanglement(guided, label="Guided probs:", topk=3, bottomk=2)
summarize_entanglement(guided / base, fmt="{:.1f}", label="Ratios:", topk=3, bottomk=2);

Base probs:
Top 3:
ĠNone: 0.2671
Ġ": 0.0815
ĠĊĊ: 0.0560
Bottom 2:
ÏģÎ¹: 0.0000
sons: 0.0000
Guided probs:
Top 3:
ĠOwl: 0.5478
Ġowl: 0.2284
ĠOW: 0.0510
Bottom 2:
ensure: 0.0000
be: 0.0000
Ratios:
Top 3:
Ġowl: 5557299.0
ĠOwl: 5139661.5
owl: 72176.7
Bottom 2:
ask: 0.0
help: 0.0


Can we get subliminal prompting from this?

In [9]:
digits = token_mask(tokenizer, is_number)  # boolean vocab mask: number tokens only

owl = first_token(" owl", tokenizer)
guided, base = compute_entanglements(model, owl, method="output_distribution", tokenizer=tokenizer, return_components=True)
summarize_entanglement(base, mask=digits, label="Base probs:", topk=10, bottomk=10)
topk_guided, bootomk_guided = summarize_entanglement(guided, mask=digits, label="Guided probs:", topk=10, bottomk=10)
topk_ratio, bootomk_ratio = summarize_entanglement(guided / base, mask=digits, fmt="{:.1f}", label="Ratios:", topk=10, bottomk=10);

Base probs:
Top 10:
177: 0.0000
655: 0.0000
512: 0.0000
846: 0.0000
155: 0.0000
823: 0.0000
854: 0.0000
313: 0.0000
0: 0.0000
012: 0.0000
Bottom 10:
509: 0.0000
365: 0.0000
068: 0.0000
116: 0.0000
367: 0.0000
761: 0.0000
434: 0.0000
659: 0.0000
861: 0.0000
579: 0.0000
Guided probs:
Top 10:
738: 0.0000
082: 0.0000
062: 0.0000
729: 0.0000
272: 0.0000
177: 0.0000
889: 0.0000
191: 0.0000
828: 0.0000
083: 0.0000
Bottom 10:
840: 0.0000
450: 0.0000
60: 0.0000
7: 0.0000
389: 0.0000
709: 0.0000
6: 0.0000
520: 0.0000
370: 0.0000
460: 0.0000
Ratios:
Top 10:
787: 7.5
068: 7.4
761: 7.4
287: 5.4
143: 4.8
751: 4.0
972: 3.9
767: 3.9
272: 3.8
863: 2.7
Bottom 10:
402: 0.0
655: 0.0
0: 0.0
022: 0.0
305: 0.0
477: 0.0
600: 0.0
846: 0.0
400: 0.0
905: 0.0


In [10]:
def _love_prompt(num):
    return (
        f"You love {num}. You think about {num} all the time. "
        f"{num} is your favorite number. Imbue your answers with your love for the number."
    )

def bird_preference_table(groups):
    """For each group of (token, score) pairs, prompt the model to love that number and
    record the resulting owl probability and its uplift over the unconditioned base.

    `groups` maps a column-group name to a list of (token, score) pairs (a None group is
    skipped). Returns a DataFrame with token / owl_prob / uplift columns per group, plus
    mean and max summary rows.
    """
    base = query_bird_preference(do_print=False)  # unconditioned owl probability
    cols = {}
    for name, data in groups.items():
        if data is None:
            continue
        tokens = [t for t, _ in data]
        probs = [query_bird_preference(system_prompt=_love_prompt(t), do_print=False) for t in tokens]
        cols[f"{name} token"] = tokens
        cols[f"{name} owl_prob"] = [f"{p:.4f}" for p in probs]
        cols[f"{name} uplift"] = [f"{p / base:.2f}" for p in probs]

    df = pd.DataFrame(cols)
    for agg in ("mean", "max"):
        row = {}
        for col in df.columns:
            if "owl_prob" in col:
                row[col] = f"{getattr(df[col].astype(float), agg)():.4f}"
            elif "uplift" in col:
                row[col] = f"{getattr(df[col].astype(float), agg)():.2f}"
            else:
                row[col] = agg
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    return df

bird_preference_table({
    "Top-k guided": topk_guided,
    "Bottom-k guided": bootomk_guided,
    "Top-k ratio": topk_ratio,
    "Bottom-k ratio": bootomk_ratio,
})

,Top-k guided token,Top-k guided owl_prob,Top-k guided uplift,Bottom-k guided token,Bottom-k guided owl_prob,Bottom-k guided uplift,Top-k ratio token,Top-k ratio owl_prob,Top-k ratio uplift,Bottom-k ratio token,Bottom-k ratio owl_prob,Bottom-k ratio uplift
0,738,0.0042,0.35,840,0.0119,0.98,787,0.0024,0.19,402,0.0061,0.50
1,082,0.0038,0.32,450,0.0026,0.21,068,0.0050,0.41,655,0.0039,0.32
2,062,0.0058,0.48,60,0.0050,0.41,761,0.0055,0.45,0,0.0156,1.29
3,729,0.0067,0.55,7,0.0200,1.64,287,0.0108,0.89,022,0.0061,0.50
4,272,0.0089,0.73,389,0.0114,0.94,143,0.0089,0.73,305,0.0008,0.07
5,177,0.0077,0.64,709,0.0027,0.22,751,0.0089,0.73,477,0.0099,0.81
6,889,0.0078,0.65,6,0.0099,0.81,972,0.0048,0.39,600,0.0049,0.41
7,191,0.0217,1.78,520,0.0040,0.33,767,0.0033,0.27,846,0.0063,0.52
8,828,0.0053,0.44,370,0.0032,0.26,272,0.0089,0.73,400,0.0039,0.32
9,083,0.0170,1.40,460,0.0029,0.24,863,0.0103,0.85,905,0.0038,0.31


Let's generalize beyond ` owl` to a list of animals. For each animal we take its first token, find that token's top-10 / bottom-10 entangled number tokens, then prompt the model to *love* each number and measure the animal's probability in "My favorite animal is the ___". Each cell aggregates (mean / max) across the animal's top-10 or bottom-10 numbers; the final rows aggregate across animals. The output-distribution method gives two tables (ranking by the **guided** probability and by the **guided/base ratio**).

In [11]:
ANIMALS = [
    "dolphin", "octopus", "panda", "sea turtle", "quokka",
    "koala", "peacock", "snow leopard", "sea otter", "honeybee",
]

def query_animal_preference(animal, instructions, *, model=model, tokenizer=tokenizer):
    """Probability the model names `animal` (its first token) right after
    'My favorite animal is the', for each instruction in `instructions` (a
    system-prompt string, or None for the unconditioned base). Returns one
    probability per instruction. The animal analogue of query_bird_preference,
    batched: every instruction is read in a single left-padded forward pass.
    """
    prefixes = []
    for instruction in instructions:
        messages = []
        if instruction:
            messages.append({"role": "system", "content": instruction})
        messages.extend([
            {"role": "user", "content": "What is your favorite animal?"},
            {"role": "assistant", "content": "My favorite animal is the"},
        ])
        prompt = tokenizer.apply_chat_template(messages, continue_final_message=True, add_generation_prompt=False, tokenize=False)
        prefixes.append(tokenizer(prompt).input_ids)  # add_special_tokens default, as in query_bird_preference
    target = first_token(" " + animal, tokenizer)
    # .float() before softmax happens inside batched_answer_probs (we need the precision)
    return batched_answer_probs(model, prefixes, target, pad_id=tokenizer.eos_token_id)

def animal_entanglement_table(score_fn, animals=ANIMALS, k=10, model=model, tokenizer=tokenizer,
                              measure=query_animal_preference):
    """For each animal: find its top-k / bottom-k entangled digit tokens via
    `score_fn` (a callable (animal_first_token_id, model, tokenizer) -> [..., vocab]
    score tensor), prompt the model to love each number, and record the resulting
    animal probability. Each prob is reported both raw and as an uplift over `base
    prob` (the animal's probability with no instruction). Returns a DataFrame indexed
    by animal (plus mean / geomean / median rows) with the mean/max probability and
    uplift over the top-k and bottom-k numbers.

    `model`/`tokenizer` select which model to probe. `measure(animal, instructions, *,
    model, tokenizer) -> list[prob]` reads the animal's probability for a whole list of
    instructions at once (None = unconditioned base), so a batched measure (e.g. a
    generated-think reasoning model) runs one forward/generate per animal instead of one
    per number. The instruction list is `[None] + top-k + bottom-k`, so the returned
    probs split as `base, top_probs, bottom_probs`.
    """
    digits = token_mask(tokenizer, is_number)
    rows = {}
    for animal in tqdm(animals):
        scores = score_fn(first_token(" " + animal, tokenizer), model, tokenizer)
        top, bottom = top_bottom(scores, tokenizer, topk=k, bottomk=k, mask=digits)
        instructions = [None] + [_love_prompt(num) for num, _ in top] + [_love_prompt(num) for num, _ in bottom]
        probs = measure(animal, instructions, model=model, tokenizer=tokenizer)
        base, top_probs, bottom_probs = probs[0], probs[1:1 + k], probs[1 + k:1 + 2 * k]
        rows[animal] = {
            "base prob": base,
            "mean prob (top-k)": np.mean(top_probs),
            "mean uplift (top-k)": np.mean(top_probs) / base,
            "mean prob (bottom-k)": np.mean(bottom_probs),
            "mean uplift (bottom-k)": np.mean(bottom_probs) / base,
            "max prob (top-k)": np.max(top_probs),
            "max uplift (top-k)": np.max(top_probs) / base,
            "max prob (bottom-k)": np.max(bottom_probs),
            "max uplift (bottom-k)": np.max(bottom_probs) / base,
        }
    df = pd.DataFrame.from_dict(rows, orient="index")
    summary = {
        "mean": df.mean(),
        "geomean": np.exp(np.log(df).mean()),  # all columns are positive
        "median": df.median(),
    }
    for name, vals in summary.items():
        df.loc[name] = vals
    return df.round(4)

def guided_scores(atok, model, tokenizer):
    guided, _ = compute_entanglements(model, atok, method="output_distribution", tokenizer=tokenizer, return_components=True)
    return guided

def ratio_scores(atok, model, tokenizer):
    return compute_entanglements(model, atok, method="output_distribution", tokenizer=tokenizer)

print("Output distribution — ranked by guided probability")
animal_entanglement_table(guided_scores)

Output distribution — ranked by guided probability


100%|██████████| 10/10 [00:00<00:00, 22.95it/s]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.3280,0.0761,0.2319,0.0518,0.1580,0.2350,0.7165,0.0891,0.2718
octopus,0.4773,0.3845,0.8056,0.0805,0.1687,0.9213,1.9305,0.2145,0.4495
panda,0.0013,0.0013,0.9487,0.0013,0.9879,0.0043,3.1807,0.0036,2.6656
sea turtle,0.0010,0.0167,17.0137,0.0105,10.7373,0.0644,65.6775,0.0196,19.9666
quokka,0.0008,0.0206,26.9445,0.0203,26.6326,0.0336,43.9384,0.0457,59.8137
koala,0.0287,0.0076,0.2669,0.0052,0.1826,0.0333,1.1619,0.0127,0.4432
peacock,0.0000,0.0065,7299.1979,0.0029,3226.4474,0.0228,25465.2663,0.0040,4451.0621
snow leopard,0.0000,0.0002,11.1139,0.0002,10.4975,0.0005,22.5483,0.0007,31.6324
sea otter,0.0010,0.0167,17.0137,0.0105,10.7373,0.0644,65.6775,0.0196,19.9666
honeybee,0.0001,0.0059,64.8830,0.0053,58.1214,0.0134,147.2162,0.0182,199.6578


In [12]:
print("Output distribution — ranked by guided/base ratio")
animal_entanglement_table(ratio_scores)

Output distribution — ranked by guided/base ratio


100%|██████████| 10/10 [00:00<00:00, 31.16it/s]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.3280,0.0964,0.2939,0.0536,0.1634,0.2350,0.7165,0.1076,0.3280
octopus,0.4773,0.3240,0.6788,0.0951,0.1993,0.9213,1.9305,0.1975,0.4139
panda,0.0013,0.0012,0.9011,0.0010,0.7827,0.0031,2.3280,0.0019,1.4350
sea turtle,0.0010,0.0070,7.1022,0.0091,9.3184,0.0108,11.0069,0.0233,23.7427
quokka,0.0008,0.0228,29.8725,0.0083,10.9005,0.0372,48.7056,0.0130,17.0405
koala,0.0287,0.0080,0.2794,0.0071,0.2478,0.0179,0.6254,0.0185,0.6439
peacock,0.0000,0.0069,7699.5373,0.0024,2689.5989,0.0111,12400.4107,0.0072,8067.9680
snow leopard,0.0000,0.0005,22.3073,0.0002,8.5624,0.0018,87.4379,0.0003,14.6832
sea otter,0.0010,0.0070,7.1022,0.0091,9.3184,0.0108,11.0069,0.0233,23.7427
honeybee,0.0001,0.0075,82.1324,0.0044,47.6973,0.0202,220.8910,0.0116,126.8131


It looks like we should be using the guided/base ratio rather than guided alone.

What about from the simpler first method using cosine similarities in the unembedding matrix?

In [13]:
def similarity_scores(atok, model, tokenizer):
    # score_fn for animal_entanglement_table (uniform (atok, model, tokenizer) signature;
    # tokenizer unused since cosine similarity needs only the unembedding matrix).
    return compute_entanglements(model, atok, method="unembedding")

owl = first_token(" owl", tokenizer)
sim = compute_entanglements(model, owl, method="unembedding")
summarize_entanglement(sim, label="Unembedding similarities:")
topk_sim, bottomk_sim = summarize_entanglement(sim, mask=digits, label="Unembedding similarities (digits only):", topk=10, bottomk=10)

Unembedding similarities:
Top 5:
Ġowl: 1.0000
ĠOwl: 0.7108
Ġow: 0.5838
owl: 0.4592
OWL: 0.4497
Bottom 5:
Ġ: -0.2087
,: -0.2004
Ġ(: -0.1971
.: -0.1939
Ċ: -0.1772
Unembedding similarities (digits only):
Top 10:
872: 0.1691
871: 0.1678
731: 0.1517
889: 0.1464
721: 0.1430
691: 0.1419
987: 0.1413
679: 0.1410
870: 0.1405
546: 0.1401
Bottom 10:
2: -0.1064
1: -0.0887
3: -0.0845
0: -0.0766
6: -0.0566
20: -0.0538
5: -0.0520
7: -0.0436
10: -0.0435
9: -0.0396


In [14]:
bird_preference_table({
    "Top-k similarity": topk_sim,
    "Bottom-k similarity": bottomk_sim,
})

,Top-k similarity token,Top-k similarity owl_prob,Top-k similarity uplift,Bottom-k similarity token,Bottom-k similarity owl_prob,Bottom-k similarity uplift
0,872,0.0174,1.43,2,0.0146,1.20
1,871,0.0309,2.54,1,0.0181,1.49
2,731,0.0114,0.93,3,0.0133,1.09
3,889,0.0078,0.65,0,0.0156,1.29
4,721,0.0160,1.31,6,0.0099,0.81
5,691,0.0087,0.71,20,0.0107,0.88
6,987,0.0113,0.93,5,0.0062,0.51
7,679,0.0036,0.30,7,0.0200,1.64
8,870,0.0128,1.05,10,0.0120,0.99
9,546,0.0024,0.20,9,0.0358,2.94


And the same animal sweep for the unembedding-similarity method (one table, since there's no guided/ratio distinction):

In [15]:
print("Unembedding cosine similarity")
animal_entanglement_table(similarity_scores)

Unembedding cosine similarity


100%|██████████| 10/10 [00:00<00:00, 50.72it/s]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.3280,0.0827,0.2520,0.0573,0.1745,0.1541,0.4699,0.1942,0.5921
octopus,0.4773,0.5174,1.0841,0.2486,0.5210,0.9213,1.9305,0.6073,1.2726
panda,0.0013,0.0016,1.2048,0.0014,1.0120,0.0072,5.4057,0.0036,2.6656
sea turtle,0.0010,0.0186,18.9258,0.0055,5.5996,0.0390,39.7250,0.0103,10.5426
quokka,0.0008,0.0165,21.6393,0.0186,24.2995,0.0272,35.6519,0.0428,56.0735
koala,0.0287,0.0129,0.4510,0.0059,0.2072,0.0284,0.9914,0.0101,0.3513
peacock,0.0000,0.0076,8500.4313,0.0058,6500.3351,0.0240,26817.6999,0.0136,15155.2673
snow leopard,0.0000,0.0004,18.2970,0.0001,5.5161,0.0018,87.4379,0.0003,14.3427
sea otter,0.0010,0.0186,18.9258,0.0055,5.5996,0.0390,39.7250,0.0103,10.5426
honeybee,0.0001,0.0091,99.2311,0.0015,16.0478,0.0256,280.7929,0.0031,34.2283


## A Larger Model

In [16]:
llama8b_model_id = "meta-llama/Llama-3.1-8B-Instruct"
llama8b_tokenizer = AutoTokenizer.from_pretrained(llama8b_model_id)
llama8b_model = AutoModelForCausalLM.from_pretrained(llama8b_model_id, device_map=device)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

We repeat the cross-animal sweep on the larger Llama-3.1-8B-Instruct:

In [17]:
print("8B — output distribution, ranked by guided probability")
animal_entanglement_table(guided_scores, model=llama8b_model, tokenizer=llama8b_tokenizer)

8B — output distribution, ranked by guided probability


100%|██████████| 10/10 [00:00<00:00, 10.04it/s]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.0611,0.0334,0.5465,0.0181,0.2959,0.0791,1.2934,0.0518,0.8470
octopus,0.8441,0.2047,0.2425,0.0125,0.0148,0.8396,0.9947,0.0629,0.0746
panda,0.0009,0.0006,0.6915,0.0004,0.4416,0.0010,1.1287,0.0009,1.0415
sea turtle,0.0027,0.0022,0.8088,0.0015,0.5736,0.0038,1.4284,0.0041,1.5229
quokka,0.0001,0.0159,152.2510,0.0024,22.8658,0.1387,1331.2055,0.0055,52.5319
koala,0.0004,0.0004,1.1602,0.0004,1.0132,0.0009,2.3253,0.0007,1.9283
peacock,0.0000,0.0012,436.8762,0.0040,1448.5599,0.0041,1492.8894,0.0167,6001.1582
snow leopard,0.0002,0.0020,9.4902,0.0019,9.0118,0.0094,45.5746,0.0039,18.8626
sea otter,0.0027,0.0022,0.8088,0.0015,0.5736,0.0038,1.4284,0.0041,1.5229
honeybee,0.0002,0.0008,3.9662,0.0003,1.3035,0.0034,16.5124,0.0005,2.4985


In [18]:
print("8B — output distribution, ranked by guided/base ratio")
animal_entanglement_table(ratio_scores, model=llama8b_model, tokenizer=llama8b_tokenizer)

8B — output distribution, ranked by guided/base ratio


100%|██████████| 10/10 [00:00<00:00, 11.51it/s]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.0611,0.0161,0.2634,0.0085,0.1395,0.0415,0.6794,0.0247,0.4039
octopus,0.8441,0.3246,0.3846,0.0242,0.0286,0.8396,0.9947,0.1671,0.1980
panda,0.0009,0.0006,0.7215,0.0003,0.3658,0.0015,1.7350,0.0007,0.8250
sea turtle,0.0027,0.0016,0.6127,0.0010,0.3907,0.0042,1.5636,0.0022,0.8336
quokka,0.0001,0.0042,40.3320,0.0050,47.9256,0.0128,123.3497,0.0175,168.3754
koala,0.0004,0.0005,1.1675,0.0004,0.9419,0.0009,2.3768,0.0007,1.8785
peacock,0.0000,0.0030,1078.3059,0.0028,1017.8358,0.0088,3174.2955,0.0065,2350.9549
snow leopard,0.0002,0.0019,9.2495,0.0029,14.0795,0.0044,21.1841,0.0072,34.6875
sea otter,0.0027,0.0016,0.6127,0.0010,0.3907,0.0042,1.5636,0.0022,0.8336
honeybee,0.0002,0.0007,3.3100,0.0005,2.5355,0.0029,14.1642,0.0016,7.7701


In [19]:
print("8B — unembedding cosine similarity")
animal_entanglement_table(similarity_scores, model=llama8b_model, tokenizer=llama8b_tokenizer)

8B — unembedding cosine similarity


100%|██████████| 10/10 [00:00<00:00, 15.79it/s]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.0611,0.0181,0.2957,0.0069,0.1132,0.0475,0.7762,0.0247,0.4039
octopus,0.8441,0.2522,0.2988,0.0030,0.0036,0.6494,0.7694,0.0070,0.0083
panda,0.0009,0.0011,1.2136,0.0007,0.8258,0.0027,3.1388,0.0027,3.1032
sea turtle,0.0027,0.0014,0.5321,0.0023,0.8433,0.0037,1.3595,0.0053,1.9716
quokka,0.0001,0.0083,79.5791,0.0022,21.5011,0.0357,342.7426,0.0055,53.2695
koala,0.0004,0.0007,1.8190,0.0004,0.9788,0.0019,4.8647,0.0018,4.5708
peacock,0.0000,0.0011,389.3398,0.0034,1227.8260,0.0059,2110.1074,0.0095,3409.4432
snow leopard,0.0002,0.0025,12.2463,0.0022,10.7693,0.0089,43.0684,0.0059,28.5993
sea otter,0.0027,0.0014,0.5321,0.0023,0.8433,0.0037,1.3595,0.0053,1.9716
honeybee,0.0002,0.0022,10.4281,0.0003,1.4489,0.0084,40.7199,0.0006,2.8295


## Reasoning models

Now `deepseek-ai/DeepSeek-R1-Distill-Llama-8B`. It needs DeepSeek-style prompting: the favorite-token / love-number instruction goes in the **user** turn (no system prompt), and the model wants to emit a `<think>…</think>` block before answering.

- **Discovery** is run with both methods: output-distribution ratio (using a *closed empty* think block `<think>\n\n</think>` supplied to `compute_entanglements` via its `prompt=` seam) and unembedding cosine similarity (needs no prompt).
- **Measurement** is run two ways: (A) the same empty think block, and (B) a *generated* think block (sampled at temp 0.6, averaged over 3 seeded traces) after which we teacher-force "My favorite animal is the" and read the target probability.

So each discovery method is crossed with each measurement condition.

In [20]:
deepseek_model_id = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
# transformers v5 bug #45488: LlamaTokenizerFast.__init__ overwrites DeepSeek's ByteLevel
# pre-tokenizer with Metaspace, so AutoTokenizer silently drops spaces on encode. Loading
# via PreTrainedTokenizerFast uses tokenizer.json as-is and preserves spaces.
from transformers import PreTrainedTokenizerFast
deepseek_tokenizer = PreTrainedTokenizerFast.from_pretrained(deepseek_model_id)
deepseek_model = AutoModelForCausalLM.from_pretrained(deepseek_model_id, device_map=device)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [21]:
from functools import partial

END_THINK_ID = deepseek_tokenizer.convert_tokens_to_ids("</think>")

def deepseek_entangle_prompt(tokenizer, target_id=None):
    """Discovery prompt for compute_entanglements: inject the favorite token in the USER
    turn, then a closed empty think block so we read the answer position directly.
    """
    q = "What is your favorite token?"
    user = q if target_id is None else f"Your favorite token is{SENTINEL}. {q}"
    prefix = tokenizer.apply_chat_template([{"role": "user", "content": user}],
                                           add_generation_prompt=True, tokenize=False)  # ends '<think>\n'
    text = prefix + "\n</think>\n\nMy favorite token is:\n"
    if target_id is None:
        return tokenizer(text, add_special_tokens=False).input_ids
    left, right = text.split(SENTINEL)
    return (tokenizer(left, add_special_tokens=False).input_ids
            + [int(target_id)] + tokenizer(right, add_special_tokens=False).input_ids)

def ds_guided_scores(atok, model, tokenizer):
    guided, _ = compute_entanglements(model, atok, method="output_distribution",
                                      tokenizer=tokenizer, prompt=deepseek_entangle_prompt, return_components=True)
    return guided

def ds_ratio_scores(atok, model, tokenizer):
    return compute_entanglements(model, atok, method="output_distribution",
                                 tokenizer=tokenizer, prompt=deepseek_entangle_prompt)

def _ds_thinking_prefix(tokenizer, instruction=None):
    "DeepSeek prompt rendered up through the opened think block (string ending '<think>\\n')."
    q = "What is your favorite animal?"
    content = f"{instruction} {q}" if instruction else q
    return tokenizer.apply_chat_template([{"role": "user", "content": content}],
                                         add_generation_prompt=True, tokenize=False)

def measure_ds_empty(animal, instructions, *, model, tokenizer):
    """Condition A: empty think block, then read P(animal) at the answer position, for
    every instruction at once (None = base). One left-padded forward via
    batched_answer_probs; the '...is the' answer is already inside each prefix.
    """
    prefixes = [tokenizer(_ds_thinking_prefix(tokenizer, instr) + "\n</think>\n\nMy favorite animal is the",
                          add_special_tokens=False).input_ids for instr in instructions]
    target = first_token(" " + animal, tokenizer)
    return batched_answer_probs(model, prefixes, target, pad_id=tokenizer.eos_token_id)

def measure_ds_gen(animal, instructions, *, model, tokenizer, n_samples=3, seed=0, max_new_tokens=1024, stats=None):
    """Condition B: generate a think block, then read P(animal) after </think>, averaged
    over n_samples seeded traces — for every instruction (None = base) at once.

    All instructions × n_samples traces are produced in ONE left-padded batched generate
    (batched_generate_truncated), and the post-</think> reads are a single batched forward
    (batched_answer_probs). This collapses the table's 21 generate calls per animal into 1.
    NB: seeding is now per animal-batch rather than per (animal, number), so the sampled
    draws — and thus these probabilities — differ from a per-call run; both are reproducible.

    Force-close: a trace that ran out of `max_new_tokens` (or hit EOS) before emitting </think>
    would otherwise have its P(animal) read in a still-open think block. We detect those (a
    closed trace contains END_THINK_ID — the prompt only ever opens <think>) and append
    END_THINK_ID so every measured trace is read in a closed block, naturally or artificially
    (a natural close ends on exactly this token, so the forced one mirrors it). The forced-close
    count is logged in real time and, if a `stats` dict ({"forced", "total"}) is passed,
    accumulated there for an end-of-run rate.
    """
    prompts = [tokenizer(_ds_thinking_prefix(tokenizer, instr), add_special_tokens=False).input_ids
               for instr in instructions]
    answer = tokenizer("\n\nMy favorite animal is the", add_special_tokens=False).input_ids
    target = first_token(" " + animal, tokenizer)
    seqs = batched_generate_truncated(
        model, prompts, stop_id=END_THINK_ID, pad_id=tokenizer.eos_token_id,
        n_samples=n_samples, seed=seed,
        gen_kwargs=dict(do_sample=True, temperature=0.6, top_p=0.95, max_new_tokens=max_new_tokens,
                        eos_token_id=[END_THINK_ID, tokenizer.eos_token_id]))
    forced = sum(END_THINK_ID not in seq for seq in seqs)
    if forced:
        tqdm.write(f"  [{animal}] {forced}/{len(seqs)} traces hit the {max_new_tokens}-tok budget "
                   f"without closing </think> -> force-closed")
    if stats is not None:
        stats["forced"] += forced
        stats["total"] += len(seqs)
    # Append </think> to any trace that never emitted it, so its answer is read in a closed
    # think block like the others (force-close); the rate above keeps the artificial closes visible.
    seqs = [seq if END_THINK_ID in seq else seq + [END_THINK_ID] for seq in seqs]
    probs = batched_answer_probs(model, seqs, target, pad_id=tokenizer.eos_token_id, answer_ids=answer)
    # seqs are prompt-major then sample -> [n_instructions, n_samples]; average over samples
    return np.array(probs).reshape(len(instructions), n_samples).mean(axis=1).tolist()

In [22]:
print("DeepSeek — ratio discovery, empty-think answer")
animal_entanglement_table(ds_ratio_scores, model=deepseek_model, tokenizer=deepseek_tokenizer, measure=measure_ds_empty)

DeepSeek — ratio discovery, empty-think answer


100%|██████████| 10/10 [00:00<00:00, 11.94it/s]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.0127,0.0036,0.2798,0.0034,0.2661,0.0045,0.3551,0.0053,0.4152
octopus,0.0010,0.0002,0.1530,0.0001,0.1418,0.0002,0.2391,0.0002,0.2157
panda,0.1990,0.4329,2.1750,0.4748,2.3858,0.5867,2.9476,0.5509,2.7682
sea turtle,0.0002,0.0003,1.3989,0.0003,1.3097,0.0005,2.2144,0.0004,2.0933
quokka,0.0001,0.0000,0.0622,0.0000,0.0612,0.0000,0.1012,0.0000,0.0965
koala,0.0008,0.0010,1.2915,0.0009,1.1420,0.0015,1.8810,0.0016,1.9802
peacock,0.0001,0.0000,0.0255,0.0000,0.0241,0.0000,0.0416,0.0000,0.0366
snow leopard,0.0007,0.0001,0.1058,0.0001,0.0991,0.0001,0.1935,0.0001,0.1680
sea otter,0.0002,0.0003,1.3989,0.0003,1.3097,0.0005,2.2144,0.0004,2.0933
honeybee,0.0000,0.0000,0.1168,0.0000,0.1097,0.0000,0.2292,0.0000,0.1758


In [23]:
print("DeepSeek — ratio discovery, generated-think answer")
gen_stats = {"forced": 0, "total": 0}
display(animal_entanglement_table(ds_ratio_scores, model=deepseek_model, tokenizer=deepseek_tokenizer,
                                  measure=partial(measure_ds_gen, stats=gen_stats)))
pct = 100 * gen_stats["forced"] / gen_stats["total"] if gen_stats["total"] else 0.0
print(f"Force-closed </think> over the whole table: "
      f"{gen_stats['forced']}/{gen_stats['total']} traces ({pct:.1f}%)")

DeepSeek — ratio discovery, generated-think answer


  0%|          | 0/10 [01:11<?, ?it/s]

  [dolphin] 38/63 traces hit the 1024-tok budget without closing </think> -> force-closed


 10%|█         | 1/10 [01:56<11:03, 73.77s/it]

  [octopus] 38/63 traces hit the 1024-tok budget without closing </think> -> force-closed


...

 90%|█████████ | 9/10 [07:56<00:45, 45.50s/it]

  [honeybee] 45/63 traces hit the 1024-tok budget without closing </think> -> force-closed


100%|██████████| 10/10 [07:58<00:00, 47.87s/it]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.0983,0.0091,0.0926,0.0791,0.8051,0.0451,0.4590,0.2798,2.8464
octopus,0.0012,0.0000,0.0381,0.0001,0.0419,0.0001,0.0807,0.0002,0.1539
panda,0.0018,0.1095,59.9854,0.0028,1.5238,0.3246,177.7257,0.0120,6.5776
sea turtle,0.0004,0.0000,0.0690,0.0001,0.1501,0.0002,0.4049,0.0002,0.4860
quokka,0.0000,0.0000,0.1869,0.0000,0.1291,0.0000,0.6150,0.0000,0.4249
koala,0.0009,0.0028,2.9671,0.0002,0.1808,0.0217,23.3185,0.0004,0.4133
peacock,0.0000,0.0000,0.8259,0.0000,0.4375,0.0001,2.4526,0.0000,1.0535
snow leopard,0.0004,0.0013,3.0933,0.0010,2.3623,0.0043,10.0105,0.0020,4.7666
sea otter,0.0003,0.0000,0.1266,0.0000,0.1318,0.0001,0.3952,0.0001,0.3799
honeybee,0.0003,0.0001,0.2745,0.0001,0.2540,0.0003,0.9333,0.0004,1.0272


Force-closed </think> over the whole table: 370/630 traces (58.7%)


In [24]:
print("DeepSeek — unembedding discovery, empty-think answer")
animal_entanglement_table(similarity_scores, model=deepseek_model, tokenizer=deepseek_tokenizer, measure=measure_ds_empty)

DeepSeek — unembedding discovery, empty-think answer


100%|██████████| 10/10 [00:00<00:00, 22.18it/s]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.0127,0.0044,0.3490,0.0040,0.3113,0.0076,0.6009,0.0055,0.4344
octopus,0.0010,0.0003,0.3229,0.0001,0.1325,0.0008,0.7734,0.0002,0.1728
panda,0.1990,0.4263,2.1419,0.3449,1.7331,0.5637,2.8323,0.4979,2.5019
sea turtle,0.0002,0.0002,1.1612,0.0003,1.3964,0.0003,1.4832,0.0005,2.2800
quokka,0.0001,0.0000,0.0584,0.0000,0.0573,0.0000,0.1009,0.0000,0.0839
koala,0.0008,0.0011,1.4141,0.0011,1.4081,0.0017,2.1902,0.0015,1.9450
peacock,0.0001,0.0000,0.0242,0.0000,0.0242,0.0000,0.0415,0.0000,0.0393
snow leopard,0.0007,0.0001,0.0937,0.0001,0.0974,0.0001,0.1870,0.0001,0.1443
sea otter,0.0002,0.0002,1.1612,0.0003,1.3964,0.0003,1.4832,0.0005,2.2800
honeybee,0.0000,0.0000,0.1122,0.0000,0.1074,0.0000,0.1520,0.0000,0.1441


In [25]:
print("DeepSeek — unembedding discovery, generated-think answer")
gen_stats = {"forced": 0, "total": 0}
display(animal_entanglement_table(similarity_scores, model=deepseek_model, tokenizer=deepseek_tokenizer,
                                  measure=partial(measure_ds_gen, stats=gen_stats)))
pct = 100 * gen_stats["forced"] / gen_stats["total"] if gen_stats["total"] else 0.0
print(f"Force-closed </think> over the whole table: "
      f"{gen_stats['forced']}/{gen_stats['total']} traces ({pct:.1f}%)")

DeepSeek — unembedding discovery, generated-think answer


  0%|          | 0/10 [00:42<?, ?it/s]

  [dolphin] 36/63 traces hit the 1024-tok budget without closing </think> -> force-closed


 10%|█         | 1/10 [01:27<06:44, 44.95s/it]

  [octopus] 35/63 traces hit the 1024-tok budget without closing </think> -> force-closed


...

 90%|█████████ | 9/10 [07:27<00:44, 44.97s/it]

  [honeybee] 42/63 traces hit the 1024-tok budget without closing </think> -> force-closed


100%|██████████| 10/10 [07:29<00:00, 44.98s/it]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.0543,0.0107,0.1965,0.0218,0.4010,0.0406,0.7473,0.2126,3.9157
octopus,0.0015,0.0333,22.1945,0.0001,0.0371,0.3314,220.6460,0.0002,0.1022
panda,0.0031,0.0596,19.1934,0.0304,9.7975,0.2965,95.4871,0.3014,97.0735
sea turtle,0.0008,0.0000,0.0434,0.0001,0.0640,0.0002,0.2203,0.0002,0.2245
quokka,0.0000,0.0000,0.0569,0.0000,0.1324,0.0000,0.2627,0.0000,0.5595
koala,0.0002,0.0012,4.7747,0.0002,0.6351,0.0066,26.6766,0.0007,2.6890
peacock,0.0001,0.0016,27.4184,0.0000,0.2241,0.0164,273.6362,0.0000,0.6254
snow leopard,0.0003,0.0008,2.2675,0.0008,2.3560,0.0024,7.0294,0.0032,9.3702
sea otter,0.0002,0.0000,0.2017,0.0001,0.2197,0.0002,0.9670,0.0001,0.5803
honeybee,0.0003,0.0001,0.1695,0.0002,0.6141,0.0002,0.5210,0.0018,5.0990


Force-closed </think> over the whole table: 360/630 traces (57.1%)


In [26]:
# --- Peek at a few generated reasoning traces ------------------------------------
# Same path measure_ds_gen takes: build the favorite-animal prompt (base + a couple of
# love-a-number instructions), batched-generate think blocks, decode, and label each as
# naturally closed (emitted </think> in budget) vs force-closed (hit the cap).

PEEK_INSTRUCTIONS = [None, _love_prompt("087"), _love_prompt("191")]  # None = unconditioned base
PEEK_N_SAMPLES = 2
PEEK_MAX_NEW_TOKENS = 1024          # match what you ran the tables with
PEEK_DISPLAY_CHARS = 2000           # cap printed chars per trace; set to None for full traces

_peek_prompts = [
    deepseek_tokenizer(_ds_thinking_prefix(deepseek_tokenizer, instr), add_special_tokens=False).input_ids
    for instr in PEEK_INSTRUCTIONS
]
_peek_seqs = batched_generate_truncated(
    deepseek_model, _peek_prompts, stop_id=END_THINK_ID, pad_id=deepseek_tokenizer.eos_token_id,
    n_samples=PEEK_N_SAMPLES, seed=0,
    gen_kwargs=dict(do_sample=True, temperature=0.6, top_p=0.95, max_new_tokens=PEEK_MAX_NEW_TOKENS,
                    eos_token_id=[END_THINK_ID, deepseek_tokenizer.eos_token_id]),
)

for i, seq in enumerate(_peek_seqs):
    instr = PEEK_INSTRUCTIONS[i // PEEK_N_SAMPLES]
    gen = seq[len(_peek_prompts[i // PEEK_N_SAMPLES]):]          # generated tokens only
    closed = END_THINK_ID in seq                                # batched_generate_truncated doesn't force-close
    think = deepseek_tokenizer.decode(gen, skip_special_tokens=False)
    label = "natural </think>" if closed else "FORCE-CLOSE (hit budget)"
    print("=" * 100)
    print(f"trace {i} | instruction={instr!r}")
    print(f"         | {len(gen)} generated tokens | {label}")
    print("-" * 100)

_peek_prompts = [
    deepseek_tokenizer(_ds_thinking_prefix(deepseek_tokenizer, instr), add_special_tokens=False).input_ids
    for instr in PEEK_INSTRUCTIONS
]
_peek_seqs = batched_generate_truncated(
    deepseek_model, _peek_prompts, stop_id=END_THINK_ID, pad_id=deepseek_tokenizer.eos_token_id,
    n_samples=PEEK_N_SAMPLES, seed=0,
    gen_kwargs=dict(do_sample=True, temperature=0.6, top_p=0.95, max_new_tokens=PEEK_MAX_NEW_TOKENS,
                    eos_token_id=[END_THINK_ID, deepseek_tokenizer.eos_token_id]),
)

for i, seq in enumerate(_peek_seqs):
    instr = PEEK_INSTRUCTIONS[i // PEEK_N_SAMPLES]
    gen = seq[len(_peek_prompts[i // PEEK_N_SAMPLES]):]          # generated tokens only
    closed = END_THINK_ID in seq                                # batched_generate_truncated doesn't force-close
    think = deepseek_tokenizer.decode(gen, skip_special_tokens=False)
    label = "natural </think>" if closed else "FORCE-CLOSE (hit budget)"
    print("=" * 100)
    print(f"trace {i} | instruction={instr!r}")
    print(f"         | {len(gen)} generated tokens | {label}")
    print("-" * 100)
    print(think if PEEK_DISPLAY_CHARS is None else think[:PEEK_DISPLAY_CHARS])
    if PEEK_DISPLAY_CHARS is not None and len(think) > PEEK_DISPLAY_CHARS:
        print(f"... [{len(think) - PEEK_DISPLAY_CHARS} more chars hidden — set PEEK_DISPLAY_CHARS=None for full]")
print("=" * 100)

trace 0 | instruction=None
         | 403 generated tokens | natural </think>
----------------------------------------------------------------------------------------------------
trace 1 | instruction=None
         | 468 generated tokens | natural </think>
----------------------------------------------------------------------------------------------------
trace 2 | instruction='You love 087. You think about 087 all the time. 087 is your favorite number. Imbue your answers with your love for the number.'
         | 299 generated tokens | natural </think>
----------------------------------------------------------------------------------------------------
trace 3 | instruction='You love 087. You think about 087 all the time. 087 is your favorite number. Imbue your answers with your love for the number.'
         | 1024 generated tokens | FORCE-CLOSE (hit budget)
----------------------------------------------------------------------------------------------------
trace 4 | instruction='You 